# Interface Plates — Structural Theory

Validates each plate variant against the load cases that govern its duty cycle.

**Computed in dedicated sims** (`sim/{plate_id}/`):
1. **Bolt pattern under fault current** — joint heating during 5-cycle bolted-fault clear; joint must stay below 6061-T6 yield-derating threshold (150°C).
2. **Thermal expansion differential** — plate (6061 alpha=23.6e-6/K) vs receiver frame (A36 alpha=11.7e-6/K); per-corner-bolt radial offset must stay within bolt clearance.

**Bounded by order-of-magnitude computation in this notebook** (last section):
3. Plate deflection under face pressure (wind dynamic pressure)
4. Stress concentration at penetrations (circular-hole Kt)
5. Wind tension per bolt

## CD plate — coolant flow & QD body OD derivation

CD is the only plate where conduit OD is **derived** rather than dictated by an upstream cable bundle. Two coolant lines (supply + return) carry the secondary loop between compute container CDU and external drycooler. QD body OD drives `power_conduit_od_mm` in `cad/specs/CD/spec.yaml`.

This is upstream of the structural loop below: it determines penetration diameter, which is then fed into the same fault-current + thermal-expansion analysis as every other plate.

### Assumptions

- **Heat load to reject**: 70 kW ± 5 kW per compute container (7× SYS-421GE-NBRT-LCC nodes × ~10 kW liquid load each: 8× HGX B200 @ 1000 W TDP + CPU/DIMM/NIC cold plates, 95% to liquid). Source: NVIDIA HGX B200 product brief; Lenovo HGX B200 1000W product guide.
- **Working fluid**: deionized water, single-phase (no PGW). cp = 4180 J/(kg·K), ρ = 1000 kg/m³.
- **Secondary loop ΔT**: 5 K ± 1 K (drycooler design point at 32 °C ambient with 5 K approach).
- **Topology**: 2 parallel hydraulic lines (supply + return), each carrying full mass flow (not split — one direction only).
- **QD candidate**: Stäubli SBX 50 (2″ class) OR equivalent (Parker QCM, CPC LQ8). Hub OD ≈ 75 mm, rated 250 LPM at <0.5 bar pressure drop.
- **Neglected**: pipe friction between CDU and CD plate (CDU pump compensates), localized turbulence at QD body (<2% of total ΔP), seasonal density variation (<0.3% over 0–40 °C).
- **Out of scope at v1**: redundant N+1 cooling line; coolant freeze protection (commercial deployments only, > 0 °C).

In [1]:
import pint
from uncertainties import ufloat

ureg = pint.UnitRegistry()

# --- Inputs (with uncertainty per assumptions cell above) ---
HEAT_LOAD = ufloat(70, 5) * ureg.kW            # 7 nodes x ~10 kW liquid load
WATER_CP = 4180.0 * ureg.J / (ureg.kg * ureg.kelvin)
WATER_DENSITY = 1000.0 * ureg.kg / ureg.m**3
DELTA_T_SECONDARY = ufloat(5, 1) * ureg.kelvin  # drycooler design point

# --- Mass flow: m_dot = Q / (cp * dT) ---
m_dot = (HEAT_LOAD / (WATER_CP * DELTA_T_SECONDARY)).to(ureg.kg / ureg.s)

# --- Volumetric flow: V_dot = m_dot / rho ---
v_dot = (m_dot / WATER_DENSITY).to(ureg.liter / ureg.minute)

# Reason: each line (supply, return) carries the FULL mass flow — they are in
# series in the loop, not parallel branches splitting it.
v_dot_per_line = v_dot

print(f"Mass flow      : {m_dot:~P}")
print(f"Total volflow  : {v_dot:~P}")
print(f"Per line       : {v_dot_per_line:~P}")


Mass flow      : 3.3+/-0.7 kg/s
Total volflow  : (2.0+/-0.4)e+02 l/min
Per line       : (2.0+/-0.4)e+02 l/min


In [2]:
# --- Expected values (feed into sim/cd/constants.py) ---
# Pin the nominal so the sim assertion has a fixed target; tolerance comes from
# uncertainty bands above (±15% on flow rate, dominated by ΔT uncertainty).
EXPECTED_FLOW_PER_LINE_LPM = 200.0      # nominal, ±15%
EXPECTED_QD_BODY_OD_MM = 75.0            # Stäubli SBX 50 / Parker QCM 2"
EXPECTED_FLOW_REL_TOL = 0.15

# --- Sanity 1: order-of-magnitude vs published references ---
# GB200 NVL72 (72 GPUs total): >700 LPM container-internal primary loop.
# Our v1 secondary loop (per-container, 7x8=56 GPUs equivalent thermal):
# expected ~200 LPM is consistent with secondary being ~1/3 the primary
# rate due to higher dT (5 K vs 1 K).
ratio_to_nvl72 = v_dot.magnitude.nominal_value / 700.0
print(f"v1 CD flow / GB200 NVL72 primary : {ratio_to_nvl72:.2f} (expect ~0.3, secondary loop)")

# --- Sanity 2: QD pressure drop margin ---
# Stäubli SBX 50 rated 250+ LPM at <0.5 bar. At nominal 200 LPM:
qd_rated_flow_lpm = 250.0
flow_fraction = v_dot.magnitude.nominal_value / qd_rated_flow_lpm
print(f"Flow / SBX 50 rated capacity     : {flow_fraction:.2f} (must be < 1.0)")

# --- Sanity 3: line velocity (rule of thumb < 3 m/s for steel pipe) ---
qd_inner_dia = 50.0 * ureg.mm  # 2" nominal -> ~50mm bore
qd_area = (3.14159 * (qd_inner_dia / 2) ** 2).to(ureg.m**2)
v_line = (m_dot.magnitude.nominal_value * ureg.kg / ureg.s / WATER_DENSITY / qd_area).to(ureg.m / ureg.s)
print(f"Line velocity at nominal flow    : {v_line:~P.2f} (rule of thumb < 3 m/s)")

print()
print(f"=> EXPECTED_FLOW_PER_LINE_LPM = {EXPECTED_FLOW_PER_LINE_LPM} ± {EXPECTED_FLOW_REL_TOL*100:.0f}%")
print(f"=> EXPECTED_QD_BODY_OD_MM     = {EXPECTED_QD_BODY_OD_MM} (Stäubli SBX 50 class)")


v1 CD flow / GB200 NVL72 primary : 0.29 (expect ~0.3, secondary loop)
Flow / SBX 50 rated capacity     : 0.80 (must be < 1.0)
Line velocity at nominal flow    : 1.71 m/s (rule of thumb < 3 m/s)

=> EXPECTED_FLOW_PER_LINE_LPM = 200.0 ± 15%
=> EXPECTED_QD_BODY_OD_MM     = 75.0 (Stäubli SBX 50 class)


## Run all plates

In [3]:
import importlib

PLATES = ["cg", "bg_ac", "bg_dc", "cd"]
results = {}
for plate in PLATES:
    constants = importlib.import_module(f"sim.{plate}.constants")
    model = importlib.import_module(f"sim.{plate}.model")
    res = model.solve()
    results[plate] = (constants, res)
    print(f"=== {plate.upper()} ===")
    print(f"  joint temp rise  : {res.joint_temp_rise.to(constants.ureg.kelvin):.2f}")
    print(f"  thermal offset   : {res.thermal_offset.to(constants.ureg.mm):.3f}")
    print()


=== CG ===
  joint temp rise  : 50.80 kelvin
  thermal offset   : 0.449 millimeter



=== BG_AC ===
  joint temp rise  : 2.78 kelvin
  thermal offset   : 0.449 millimeter

=== BG_DC ===
  joint temp rise  : 3.21 kelvin
  thermal offset   : 0.449 millimeter



=== CD ===
  joint temp rise  : 0.09 kelvin
  thermal offset   : 0.449 millimeter



## Verdicts table

In [4]:
print(f"{'plate':<6} {'fault rise (K)':<18} {'fault verdict':<14} {'thermal off (mm)':<18} {'thermal margin (mm)':<22} {'thermal verdict':<14}")
print("-" * 100)
for plate_name, (consts, res) in results.items():
    rise_k = res.joint_temp_rise.to(consts.ureg.kelvin).magnitude
    fault_verdict = "PASS" if rise_k < (consts.JOINT_TEMP_THRESHOLD_C - consts.T_AMBIENT_FAULT_C) else "FAIL"

    offset_mm = res.thermal_offset.to(consts.ureg.mm).magnitude
    clearance_mm = consts.BOLT_CLEARANCE_RADIAL.to(consts.ureg.mm).magnitude
    margin_mm = clearance_mm - offset_mm
    thermal_verdict = "PASS" if margin_mm > 0 else "FAIL"

    print(f"{plate_name:<6} {rise_k:<18.2f} {fault_verdict:<14} {offset_mm:<18.3f} {margin_mm:<22.3f} {thermal_verdict:<14}")


plate  fault rise (K)     fault verdict  thermal off (mm)   thermal margin (mm)    thermal verdict
----------------------------------------------------------------------------------------------------
cg     50.80              PASS           0.449              0.051                  PASS          
bg_ac  2.78               PASS           0.449              0.051                  PASS          
bg_dc  3.21               PASS           0.449              0.051                  PASS          
cd     0.09               PASS           0.449              0.051                  PASS          


## Design risk mitigation — corner-slot adoption

Round corner-bolt holes leave a ~0.05 mm thermal margin (per the verdicts table above), which ISO 2768-m fab tolerance (±0.1 mm) consumes entirely. **Adopted mitigation: radially-slotted corner bolt holes.** Edge-midpoint bolts stay round (their radial offset is much smaller — they sit on the symmetry axes, not the diagonal).

Slot length derivation:

$$L_{slot} = D_{hole} + 2 \cdot (\delta_{thermal} + \delta_{fab} + \delta_{margin})$$

where $\delta_{thermal}$ is per-corner radial offset, $\delta_{fab}$ is hole-position tolerance per ISO 2768-m, and $\delta_{margin}$ is the engineering safety factor.

Slot orientation: each corner slot points radially toward the bolt-pattern center. The bolt rides axially (clamped via washer); slot absorbs radial growth/shrink.

In [5]:
# --- Slot length derivation (commercial CG geometry, applies to all plates) ---
# Same Δα + diagonal + ΔT across the 4-plate fleet → same slot length.
HOLE_DIAMETER_MM = 11.0          # M10 clearance hole
THERMAL_OFFSET_MM = 0.449        # per-corner radial offset at full ΔT (verdicts table)
FAB_TOLERANCE_MM = 0.1           # ISO 2768-m hole position
SAFETY_MARGIN_MM = 0.2           # engineering buffer

slot_length_mm = HOLE_DIAMETER_MM + 2 * (THERMAL_OFFSET_MM + FAB_TOLERANCE_MM + SAFETY_MARGIN_MM)
print(f"Computed slot length        : {slot_length_mm:.2f} mm")
print("Fab-spec slot length        : 13.00 mm  (rounded up)")

# Sanity: slot should accommodate full ±thermal offset + fab tol + margin in radial direction.
slot_radial_extra = (13.0 - HOLE_DIAMETER_MM) / 2  # mm beyond hole center on each side
budget_each_side = THERMAL_OFFSET_MM + FAB_TOLERANCE_MM + SAFETY_MARGIN_MM
print(f"Slot radial extra per side  : {slot_radial_extra:.2f} mm")
print(f"Budget per side             : {budget_each_side:.2f} mm  (must be ≤ slot extra)")
assert slot_radial_extra >= budget_each_side, "13 mm slot insufficient — recompute"

# Defense-extreme sanity: ΔT = 111 K (-40 to +71 °C MIL-STD-810H), what's the offset?
defense_offset_mm = THERMAL_OFFSET_MM * 111 / 85
defense_budget = defense_offset_mm + FAB_TOLERANCE_MM + SAFETY_MARGIN_MM
print(f"Defense ΔT=111K offset      : {defense_offset_mm:.3f} mm; budget: {defense_budget:.2f} mm")
print(f"Defense headroom in 13 mm   : {slot_radial_extra - defense_budget:.2f} mm  (must be ≥ 0)")


Computed slot length        : 12.50 mm
Fab-spec slot length        : 13.00 mm  (rounded up)
Slot radial extra per side  : 1.00 mm
Budget per side             : 0.75 mm  (must be ≤ slot extra)
Defense ΔT=111K offset      : 0.586 mm; budget: 0.89 mm
Defense headroom in 13 mm   : 0.11 mm  (must be ≥ 0)


In [6]:
# Sanity 1: fault energy <<< BESS capacity. Should be ~1e-7 ratio.
for plate_name, (consts, _res) in results.items():
    n_eff = consts.BOLT_COUNT / 2
    i_worst = (consts.FAULT_CURRENT.magnitude.nominal_value + consts.FAULT_CURRENT.magnitude.std_dev) * consts.ureg.kA
    r_worst = (consts.R_JOINT_PER_BOLT.magnitude.nominal_value + 2 * consts.R_JOINT_PER_BOLT.magnitude.std_dev) * consts.ureg.microohm
    energy = (i_worst**2 * r_worst / n_eff * consts.FAULT_DURATION).to(consts.ureg.J)
    ratio = (energy / (1.9 * consts.ureg.MWh)).to(consts.ureg.dimensionless)
    print(f"{plate_name.upper()} fault energy: {energy:.1f} ; ratio to 1.9 MWh BESS: {ratio:.2e}")


CG fault energy: 5434.9 joule ; ratio to 1.9 MWh BESS: 7.95e-07 dimensionless
BG_AC fault energy: 297.0 joule ; ratio to 1.9 MWh BESS: 4.34e-08 dimensionless
BG_DC fault energy: 343.8 joule ; ratio to 1.9 MWh BESS: 5.03e-08 dimensionless
CD fault energy: 9.7 joule ; ratio to 1.9 MWh BESS: 1.42e-09 dimensionless


In [7]:
# Sanity 2: differential expansion order of magnitude.
for plate_name, (consts, _res) in results.items():
    delta_alpha = consts.PLATE_ALPHA - consts.FRAME_ALPHA
    per_meter = (delta_alpha * 1000 * consts.ureg.mm * 85 * consts.ureg.kelvin).to(consts.ureg.mm)
    print(f"{plate_name.upper()} differential expansion per meter at deltaT=85K: {per_meter:.3f}")


CG differential expansion per meter at deltaT=85K: 1.012 millimeter
BG_AC differential expansion per meter at deltaT=85K: 1.012 millimeter
BG_DC differential expansion per meter at deltaT=85K: 1.012 millimeter
CD differential expansion per meter at deltaT=85K: 1.012 millimeter


## Other load cases — bounded by computation

Three load cases without dedicated sims because they're analytically bounded far below failure thresholds. Worst-case input: 50 m/s sustained wind (≈ 180 km/h, hurricane Cat 3) — not a survival design point, but a conservative ceiling for service-life loading.

Computed once for the commercial CG geometry (640×840×6mm 6061-T6). Defense plates (10mm 5083-H116) are stiffer and have similar yield (~228 MPa for H116) — defense bounds are tighter than commercial by inspection.

In [8]:
# --- Geometry + material (commercial CG) ---
PLATE_B = 640.0 * ureg.mm           # short side
PLATE_A = 840.0 * ureg.mm           # long side
PLATE_T = 6.0 * ureg.mm             # commercial thickness
E_6061 = 68.9 * ureg.GPa            # 6061-T6 modulus
SIGMA_YIELD_6061 = 276.0 * ureg.MPa # 6061-T6 yield (ASM Aluminum Handbook)

# --- Worst-case face load: 50 m/s wind dynamic pressure ---
# q = ½·ρ·v² (Bernoulli). Air density at sea-level ISA = 1.225 kg/m³.
RHO_AIR = 1.225 * ureg.kg / ureg.m**3
WIND_VELOCITY = 50.0 * ureg.m / ureg.s
WIND_PRESSURE = (0.5 * RHO_AIR * WIND_VELOCITY**2).to(ureg.kPa)
print(f"Wind dynamic pressure (50 m/s)   : {WIND_PRESSURE:~P.3f}")


Wind dynamic pressure (50 m/s)   : 1.531 kPa


In [9]:
# --- Load case 3: plate deflection under uniform face pressure ---
# Closed-form: simply-supported rectangular plate, uniform pressure load
# (Timoshenko & Woinowsky-Krieger, "Theory of Plates and Shells", thin-plate
# theory for rectangular plates with all 4 edges simply supported).
#
#   δ_max = α · p · b⁴ / (E · t³)
#   σ_max = β · p · b² / t²
#
# α and β depend on aspect ratio a/b. For aspect 840/640 = 1.31, interpolate
# between tabulated a/b=1.2 and a/b=1.4: α ≈ 0.0826, β ≈ 0.4863.
ALPHA_DEFL = 0.0826
BETA_STRESS = 0.4863

deflection = (ALPHA_DEFL * WIND_PRESSURE * PLATE_B**4 / (E_6061 * PLATE_T**3)).to(ureg.mm)
sigma_bending = (BETA_STRESS * WIND_PRESSURE * PLATE_B**2 / PLATE_T**2).to(ureg.MPa)
yield_util = (sigma_bending / SIGMA_YIELD_6061).to(ureg.dimensionless).magnitude

print(f"Plate deflection at 50 m/s wind  : {deflection:~P.2f}")
print(f"Plate bending stress             : {sigma_bending:~P.2f}")
print(f"Yield utilization                : {yield_util * 100:.2f} %")
verdict = "PASS" if deflection.magnitude < 5 and yield_util < 0.25 else "FAIL"
print(f"Verdict (δ<5mm AND σ<25% yield)  : {verdict}")


Plate deflection at 50 m/s wind  : 1.43 mm
Plate bending stress             : 8.47 MPa
Yield utilization                : 3.07 %
Verdict (δ<5mm AND σ<25% yield)  : PASS


In [10]:
# --- Load case 4: stress concentration at penetrations ---
# Peterson's "Stress Concentration Factors" — circular hole in a finite-width
# plate under uniaxial in-plane stress. Kt limit for d/W → 0 is 3 (infinite
# plate); for d/W = 0.114 (CG worst case: Ø73 power conduit / 640 short side),
# Kt rises slightly. Use 3.05 as conservative bound.
#
# In-plane stress in the plate body is dominated by bending from the face
# pressure case above (8.5 MPa). Differential thermal expansion is handled by
# ADR-015 corner slots, transferred to bolt clearance, not into plate body.
KT_HOLE = 3.05

sigma_at_hole = sigma_bending * KT_HOLE
yield_util_hole = (sigma_at_hole / SIGMA_YIELD_6061).to(ureg.dimensionless).magnitude

print(f"Bending stress in plate body     : {sigma_bending:~P.2f}")
print(f"Kt at circular penetration       : {KT_HOLE}")
print(f"Peak stress at hole edge         : {sigma_at_hole:~P.2f}")
print(f"Yield utilization at hole        : {yield_util_hole * 100:.2f} %")
verdict_hole = "PASS" if yield_util_hole < 0.25 else "FAIL"
print(f"Verdict (peak σ < 25% yield)     : {verdict_hole}")


Bending stress in plate body     : 8.47 MPa
Kt at circular penetration       : 3.05
Peak stress at hole edge         : 25.84 MPa
Yield utilization at hole        : 9.36 %
Verdict (peak σ < 25% yield)     : PASS


In [11]:
# --- Load case 5: wind tension per bolt ---
# Total wind force on plate face → distributed to 8 bolts as tension.
# Conservative: assume uniform sharing (corner bolts actually carry slightly
# more under bending than midpoints, but uniform is a good first bound for
# total tension capacity).
PLATE_FACE_AREA = (PLATE_A * PLATE_B).to(ureg.m**2)
WIND_TOTAL_FORCE = (WIND_PRESSURE * PLATE_FACE_AREA).to(ureg.N)
N_BOLTS = 8
PER_BOLT_TENSION = (WIND_TOTAL_FORCE / N_BOLTS).to(ureg.N)

# M10 grade 8.8 minimum proof load per ISO 898-1: F_p = A_s · f_p
# where A_s = 58 mm² (M10 stress area) and f_p = 580 MPa (8.8 proof stress).
# Typical assembly preload = 0.7 × proof = ~24 kN per ISO 898-1 / VDI 2230.
M10_88_PRELOAD = 24.0 * ureg.kN
preload_util = (PER_BOLT_TENSION / M10_88_PRELOAD).to(ureg.dimensionless).magnitude

print(f"Plate face area                  : {PLATE_FACE_AREA:~P.3f}")
print(f"Total wind force on plate        : {WIND_TOTAL_FORCE:~P.0f}")
print(f"Per-bolt tension (8 bolts)       : {PER_BOLT_TENSION:~P.0f}")
print(f"M10 8.8 typical preload          : {M10_88_PRELOAD:~P}")
print(f"Wind tension / preload           : {preload_util * 100:.2f} %")
verdict_bolt = "PASS" if preload_util < 0.05 else "FAIL"
print(f"Verdict (wind tension < 5%)      : {verdict_bolt}")


Plate face area                  : 0.538 m²
Total wind force on plate        : 823 N
Per-bolt tension (8 bolts)       : 103 N
M10 8.8 typical preload          : 24.0 kN
Wind tension / preload           : 0.43 %
Verdict (wind tension < 5%)      : PASS


In [12]:
# --- Bounded-by-computation summary ---
print(f"{'Load case':<35} {'Result':<22} {'% of limit':<14} {'Verdict':<6}")
print("-" * 80)
print(f"{'Plate deflection (50 m/s wind)':<35} {f'{deflection.magnitude:.2f} mm':<22} "
      f"{'(δ < 5 mm)':<14} {verdict:<6}")
print(f"{'Plate bending stress':<35} {f'{sigma_bending.magnitude:.2f} MPa':<22} "
      f"{f'{yield_util * 100:.2f} %':<14} {verdict:<6}")
print(f"{'Stress concentration at hole':<35} {f'{sigma_at_hole.magnitude:.2f} MPa':<22} "
      f"{f'{yield_util_hole * 100:.2f} %':<14} {verdict_hole:<6}")
print(f"{'Wind tension per bolt':<35} {f'{PER_BOLT_TENSION.magnitude:.0f} N':<22} "
      f"{f'{preload_util * 100:.2f} %':<14} {verdict_bolt:<6}")


Load case                           Result                 % of limit     Verdict
--------------------------------------------------------------------------------
Plate deflection (50 m/s wind)      1.43 mm                (δ < 5 mm)     PASS  
Plate bending stress                8.47 MPa               3.07 %         PASS  
Stress concentration at hole        25.84 MPa              9.36 %         PASS  
Wind tension per bolt               103 N                  0.43 %         PASS  
